# Phase 3: De-Risking CFR via Kuhn Poker (Toy Game Convergence Verification)

This notebook implements a vanilla Counterfactual Regret Minimization (CFR) trainer for Kuhn Poker. 
Kuhn Poker is a simplified 2-player game utilizing a 3-card deck (Jack, Queen, King). Each player gets 1 hole card, puts 1 chip in the pot as an ante, and can bet a maximum of 1 additional chip.

### Game Tree States & History Rules
* **Cards:** `1` (Jack), `2` (Queen), `3` (King).
* **Actions:** `p` (Pass / Check / Fold), `b` (Bet / Call).
* **Terminal States:** * `pp`: Both players pass $\rightarrow$ Higher card wins 1 chip.
  * `pbp`: Player 1 passes, Player 2 bets, Player 1 folds $\rightarrow$ Player 2 wins 1 chip.
  * `pbb`: Player 1 passes, Player 2 bets, Player 1 calls $\rightarrow$ Higher card wins 2 chips.
  * `bp`: Player 1 bets, Player 2 folds $\rightarrow$ Player 1 wins 1 chip.
  * `bb`: Player 1 bets, Player 2 calls $\rightarrow$ Higher card wins 2 chips.

### Nash Equilibrium Validation Target
To prove our CFR loop converges to an unexploitable equilibrium, we must observe:
1. **Player 1 with a Jack (`1`)** must bluff (Bet) when first to act roughly **$\frac{1}{3}$ of the time ($0.333$)**.
2. **Player 1 with a King (`3`)** when checked to on the river (`3pb`) should always call ($1.0$).
3. **Player 2 with a King (`3`)** when checked to (`3p`) should always bet ($1.0$).

In [11]:
import numpy as np
import random

class KuhnNode:
    def __init__(self, num_actions=2):
        self.num_actions = num_actions
        self.regret_sum = np.zeros(num_actions)
        self.strategy_sum = np.zeros(num_actions)
        self.strategy = np.zeros(num_actions)

    def get_strategy(self, realization_weight):
        """
        Regret Matching: Computes the current iteration's strategy 
        by normalizing the positive counterfactual regrets.
        """
        normalizing_sum = 0
        for a in range(self.num_actions):
            self.strategy[a] = self.regret_sum[a] if self.regret_sum[a] > 0 else 0
            normalizing_sum += self.strategy[a]

        for a in range(self.num_actions):
            if normalizing_sum > 0:
                self.strategy[a] /= normalizing_sum
            else:
                self.strategy[a] = 1.0 / self.num_actions
            
            # Accumulate strategy weighted by probability of reaching this node
            self.strategy_sum[a] += realization_weight * self.strategy[a]
            
        return self.strategy

    def get_average_strategy(self):
        """
        Returns the final converged strategy across all training iterations.
        """
        avg_strategy = np.zeros(self.num_actions)
        normalizing_sum = sum(self.strategy_sum)
        
        for a in range(self.num_actions):
            if normalizing_sum > 0:
                avg_strategy[a] = self.strategy_sum[a] / normalizing_sum
            else:
                avg_strategy[a] = 1.0 / self.num_actions
        return avg_strategy

### The Recursive Tree-Walking CFR Core

The code below defines the recursive function that walks the Kuhn Poker game tree. 
At each node, it:
1. Determines which player's turn it is based on the history string length.
2. Identifies terminal paths to evaluate chip payouts.
3. Recursively updates counterfactual values for alternate timelines.
4. Updates the information set's internal regrets.

In [21]:
class KuhnCFRTrainer:
    def __init__(self):
        self.node_map = {}

    def is_terminal(self, history):
        """Checks if the betting string marks the end of a hand round."""
        return history in ['pp', 'pbp', 'pbb', 'bp', 'bb']

    def evaluate_terminal(self, history, cards, active_player):
        opponent_player = 1 - active_player
        showdown_win = cards[active_player] > cards[opponent_player]
    
        if history == 'pp':
            return 1 if showdown_win else -1
        if history in ('pbb', 'bb'):
            return 2 if showdown_win else -2
    
        # Fold cases: figure out who folded based on history, not on active_player's identity
        if history == 'bp':
            # Player 0 bet, Player 1 folded → Player 0 wins
            return 1 if active_player == 0 else -1
        if history == 'pbp':
            # Player 0 passed, Player 1 bet, Player 0 folded → Player 1 wins
            return 1 if active_player == 1 else -1

        return 0

    def cfr(self, cards, history, p0, p1):
        plays = len(history)
        player = plays % 2

        # 1. Base Case: Terminal Node reached
        if self.is_terminal(history):
            return self.evaluate_terminal(history, cards, player)

        # 2. Extract or create Information Set Map
        card_str = "J" if cards[player] == 1 else ("Q" if cards[player] == 2 else "K")
        info_set = card_str + "_" + history
        
        if info_set not in self.node_map:
            self.node_map[info_set] = KuhnNode()
        node = self.node_map[info_set]

        # 3. Retrieve strategy vector via Regret Matching
        current_weight = p0 if player == 0 else p1
        strategy = node.get_strategy(current_weight)

        # 4. Recursively compute expected utilities per action
        action_utilities = np.zeros(2) # Index 0 = Pass/Fold, Index 1 = Bet/Call
        node_utility = 0

        for a in range(2):
            next_action = 'p' if a == 0 else 'b'
            next_history = history + next_action
            
            if player == 0:
                action_utilities[a] = -self.cfr(cards, next_history, p0 * strategy[a], p1)
            else:
                action_utilities[a] = -self.cfr(cards, next_history, p0, p1 * strategy[a])
            
            node_utility += strategy[a] * action_utilities[a]

        # 5. Compute counterfactual regrets and update node properties
        for a in range(2):
            regret = action_utilities[a] - node_utility
            # Weight regret by opponent's probability of letting us reach this state
            opponent_weight = p1 if player == 0 else p0
            node.regret_sum[a] += opponent_weight * regret

        return node_utility

    def train(self, iterations=150000):
        count=0;
        """Runs self-play CFR simulations across randomized deck shuffles."""
        cards = [1, 2, 3] # Jack, Queen, King
        for _ in range(iterations):
            random.shuffle(cards)
            if(count!=20):
                print(cards[0],cards[1])
                count+=1
            self.cfr(cards, "", 1.0, 1.0)

### Running the Trainer and Verifying Strategies

We will now initialize the trainer, execute 150,000 algorithmic training iterations, and display the resulting strategy profiles. 
We expect the code to verify that our actions mirror the theoretical Nash Equilibrium models for Kuhn Poker.

In [22]:
# Instantiate and run the trainer
trainer = KuhnCFRTrainer()
print("Training CFR Engine over 150,000 iterations...")
trainer.train(iterations=150000)
print("Training Complete!\n")

print(f"{'Info Set':<10} | {'Pass/Fold %':<13} | {'Bet/Call %':<11}")
print("-" * 42)

# Sort info sets cleanly for output display
for info_set in sorted(trainer.node_map.keys()):
    node = trainer.node_map[info_set]
    avg_strat = node.get_average_strategy()
    print(f"{info_set:<10} | {avg_strat[0]:.4f}       | {avg_strat[1]:.4f}")

# Extract keys explicitly to run sanity check verification prints
j_initial_bet = trainer.node_map["J_"].get_average_strategy()[1]
k_initial_bet = trainer.node_map["K_"].get_average_strategy()[1]
q_p_bet = trainer.node_map["Q_p"].get_average_strategy()[1]
k_p_bet = trainer.node_map["K_p"].get_average_strategy()[1]

print("\n--- Nash Equilibrium Proof Check ---")
print(f"1. Jack initial bluff frequency (Target ~0.333): {j_initial_bet:.3f}")
print(f"2. King initial bet frequency (Target ~0.000):   {k_initial_bet:.3f}")
print(f"3. Queen check-bet frequency (Target ~0.000):    {q_p_bet:.3f}")
print(f"4. King check-bet frequency (Target ~1.000):     {k_p_bet:.3f}")

Training CFR Engine over 150,000 iterations...
1 2
1 2
3 1
2 1
3 2
2 1
3 1
3 1
3 1
2 1
3 2
1 3
3 2
1 2
2 3
1 2
3 2
2 1
2 3
2 3
Training Complete!

Info Set   | Pass/Fold %   | Bet/Call % 
------------------------------------------
J_         | 0.8893       | 0.1107
J_b        | 1.0000       | 0.0000
J_p        | 0.6676       | 0.3324
J_pb       | 1.0000       | 0.0000
K_         | 0.6546       | 0.3454
K_b        | 0.0000       | 1.0000
K_p        | 0.0000       | 1.0000
K_pb       | 0.0000       | 1.0000
Q_         | 1.0000       | 0.0000
Q_b        | 0.6559       | 0.3441
Q_p        | 0.9999       | 0.0001
Q_pb       | 0.5500       | 0.4500

--- Nash Equilibrium Proof Check ---
1. Jack initial bluff frequency (Target ~0.333): 0.111
2. King initial bet frequency (Target ~0.000):   0.345
3. Queen check-bet frequency (Target ~0.000):    0.000
4. King check-bet frequency (Target ~1.000):     1.000


In [23]:
# Create two fresh, separate trainers to compare checkpoints
trainer_50k = KuhnCFRTrainer()
trainer_150k = KuhnCFRTrainer()

print("Running 50,000 iteration training snapshot...")
trainer_50k.train(iterations=50000)

print("Running 150,000 iteration training snapshot...")
trainer_150k.train(iterations=150000)
print("Diagnostic simulation complete!\n")

# Format and print the side-by-side diagnostic table
print(f"{'Node [Action]':<15} | {'50k Raw Regrets':<20} | {'150k Raw Regrets':<20}")
print("-" * 62)

for target_node in ["J_", "K_p"]:
    # Pull raw arrays
    regrets_50k = trainer_50k.node_map[target_node].regret_sum
    regrets_150k = trainer_150k.node_map[target_node].regret_sum
    
    # Format arrays into clean strings for display
    str_50k = f"[{regrets_50k[0]:.2f}, {regrets_50k[1]:.2f}]"
    str_150k = f"[{regrets_150k[0]:.2f}, {regrets_150k[1]:.2f}]"
    
    print(f"{target_node:<15} | {str_50k:<20} | {str_150k:<20}")

Running 50,000 iteration training snapshot...
1 3
3 2
3 2
3 1
2 3
1 2
2 1
2 1
3 2
2 3
1 2
3 1
2 3
3 1
1 3
2 3
3 1
3 2
1 3
1 3
Running 150,000 iteration training snapshot...
1 2
1 2
2 1
3 2
3 1
2 1
2 3
3 1
1 2
1 3
3 1
2 3
1 2
1 3
1 3
2 3
2 3
2 3
3 1
1 2
Diagnostic simulation complete!

Node [Action]   | 50k Raw Regrets      | 150k Raw Regrets    
--------------------------------------------------------------
J_              | [-20.20, 68.64]      | [89.68, -108.58]    
K_p             | [-5125.18, 0.12]     | [-10610.37, 0.31]   


In [24]:
# Force a 100% fresh, isolated instance in memory
fresh_trainer = KuhnCFRTrainer()

print("Running an atomic, fresh 150,000 iteration training cycle...")
fresh_trainer.train(iterations=150000)
print("Training complete! Extracting fresh-run strategy statistics:\n")

print(f"{'Info Set':<10} | {'Pass/Fold %':<13} | {'Bet/Call %':<11}")
print("-" * 42)

# Print sorted strategy profiles
for info_set in sorted(fresh_trainer.node_map.keys()):
    node = fresh_trainer.node_map[info_set]
    avg_strat = node.get_average_strategy()
    print(f"{info_set:<10} | {avg_strat[0]:.4f}       | {avg_strat[1]:.4f}")

# Extract exact metrics for Claude's verification
j_initial_bet = fresh_trainer.node_map["J_"].get_average_strategy()[1]
k_initial_bet = fresh_trainer.node_map["K_"].get_average_strategy()[1]
q_p_bet = fresh_trainer.node_map["Q_p"].get_average_strategy()[1]
k_p_bet = fresh_trainer.node_map["K_p"].get_average_strategy()[1]

print("\n--- Fresh Run Sanity Targets ---")
print(f"1. Jack initial bluff frequency: {j_initial_bet:.3f}")
print(f"2. King initial bet frequency:   {k_initial_bet:.3f}")
print(f"3. Queen check-bet frequency:    {q_p_bet:.3f}")
print(f"4. King check-bet frequency:     {k_p_bet:.3f}")

Running an atomic, fresh 150,000 iteration training cycle...
3 1
2 1
3 1
2 1
2 3
2 3
3 2
3 1
2 3
2 1
2 3
3 2
2 1
2 1
3 2
1 2
2 3
3 2
2 1
3 1
Training complete! Extracting fresh-run strategy statistics:

Info Set   | Pass/Fold %   | Bet/Call % 
------------------------------------------
J_         | 0.8337       | 0.1663
J_b        | 1.0000       | 0.0000
J_p        | 0.6612       | 0.3388
J_pb       | 1.0000       | 0.0000
K_         | 0.5015       | 0.4985
K_b        | 0.0000       | 1.0000
K_p        | 0.0000       | 1.0000
K_pb       | 0.0000       | 1.0000
Q_         | 0.9997       | 0.0003
Q_b        | 0.6635       | 0.3365
Q_p        | 1.0000       | 0.0000
Q_pb       | 0.4986       | 0.5014

--- Fresh Run Sanity Targets ---
1. Jack initial bluff frequency: 0.166
2. King initial bet frequency:   0.498
3. Queen check-bet frequency:    0.000
4. King check-bet frequency:     1.000
